In [1]:
import numpy as np
import cv2

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
painting = "/content/painting.jpg"
painting2 = "/content/painting2.jpg"
painting3 = "/content/painting3.png"
mask1 = "/content/mask1.jpg"
mask2 = "/content/mask2.jpg"


In [12]:
def preprocess_image(path):

    img = Image.open(path).convert("RGB")

    img = img.resize((128,128))

    img = np.array(img)

    img = img / 255.0   # normalize

    img = np.expand_dims(img, axis=0)

    return img

In [13]:
cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(128,128,3)),

    tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu')  # feature vector
])

In [14]:
cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
dummy = np.zeros((1,128,128,3))
cnn.predict(dummy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step


array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
      dtype=float32)

In [16]:
def import_features(image_path):

    img = preprocess_image(image_path)

    features = cnn.predict(img)

    return features

In [17]:
painting_features_cnn = import_features(painting)
mask1_features_cnn = import_features(mask1)
mask2_features_cnn = import_features(mask2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

painting_feat = import_features(painting)
painting2_feat = import_features(painting2)
painting3_feat = import_features(painting3)
mask1_feat = import_features(mask1)
mask2_feat = import_features(mask2)

sim1 = cosine_similarity(painting_feat, mask1_feat)
sim2 = cosine_similarity(painting_feat, mask2_feat)
sim3 = cosine_similarity(painting2_feat, mask1_feat)
sim4 = cosine_similarity(painting2_feat, mask2_feat)
sim5 = cosine_similarity(painting3_feat, mask1_feat)
sim6 = cosine_similarity(painting3_feat, mask2_feat)

print("Painting vs Mask1:", sim1[0][0])
print("Painting vs Mask2:", sim2[0][0])
print("Painting2 vs Mask1:", sim3[0][0])
print("Painting2 vs Mask2:", sim4[0][0])
print("Painting3 vs Mask1:", sim5[0][0])
print("Painting3 vs Mask2:", sim6[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Painting vs Mask1: 0.8910605
Painting vs Mask2: 0.886716
Painting2 vs Mask1: 0.8994606
Painting2 vs Mask2: 0.92702365
Painting3 vs Mask1: 0.81955206
Painting3 vs Mask2: 0.8862825


In [19]:
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


In [20]:
def extract_features(img_path):

    # Load image
    img = image.load_img(img_path, target_size=(224,224))

    # Convert to array
    img_array = image.img_to_array(img)

    # Expand dimensions
    img_array = np.expand_dims(img_array, axis=0)

    # Preprocess for ResNet
    img_array = preprocess_input(img_array)

    # Extract features
    features = model.predict(img_array)

    return features


In [21]:
painting_features = extract_features(painting)
painting2_features = extract_features(painting2)
painting3_features = extract_features(painting3)
mask1_features = extract_features(mask1)
mask2_features = extract_features(mask2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step


In [22]:
similarity_mask1 = cosine_similarity(painting_features, mask1_features)
similarity_mask2 = cosine_similarity(painting_features, mask2_features)

similarity_mask3 = cosine_similarity(painting2_features, mask1_features)
similarity_mask4 = cosine_similarity(painting2_features, mask2_features)

similarity_mask5 = cosine_similarity(painting3_features, mask1_features)
similarity_mask6 = cosine_similarity(painting3_features, mask2_features)

In [23]:
# Print results
print("Similarity between Painting1 and Mask1:", similarity_mask1[0][0])
print("Similarity between Painting1 and Mask2:", similarity_mask2[0][0])
# Print results
print("Similarity between Painting2 and Mask1:", similarity_mask3[0][0])
print("Similarity between Painting2 and Mask2:", similarity_mask4[0][0])
# Print results
print("Similarity between Painting3 and Mask1:", similarity_mask5[0][0])
print("Similarity between Painting3 and Mask2:", similarity_mask6[0][0])

Similarity between Painting1 and Mask1: 0.24939954
Similarity between Painting1 and Mask2: 0.30347854
Similarity between Painting2 and Mask1: 0.5453727
Similarity between Painting2 and Mask2: 0.6076472
Similarity between Painting3 and Mask1: 0.39417583
Similarity between Painting3 and Mask2: 0.46698278


In [24]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [25]:
import torch


# Function to extract embedding
def get_embedding(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model.vision_model(**inputs)

    # Extract pooled embedding
    embedding = outputs.pooler_output

    return embedding.cpu().numpy()


In [26]:
# Get embeddings
painting_emb = get_embedding(painting)
painting2_emb = get_embedding(painting2)
painting3_emb = get_embedding(painting3)
mask1_emb = get_embedding(mask1)
mask2_emb = get_embedding(mask2)

In [27]:
# Compute similarity
sim1 = cosine_similarity(painting_emb, mask1_emb)
sim2 = cosine_similarity(painting_emb, mask2_emb)
sim3 = cosine_similarity(painting2_emb, mask1_emb)
sim4 = cosine_similarity(painting2_emb, mask2_emb)
sim5 = cosine_similarity(painting3_emb, mask1_emb)
sim6 = cosine_similarity(painting3_emb, mask2_emb)


In [28]:
print("Similarity (Painting1 vs Mask1):", sim1[0][0])
print("Similarity (Painting1 vs Mask2):", sim2[0][0])
print("Similarity (Painting2 vs Mask1):", sim3[0][0])
print("Similarity (Painting2 vs Mask2):", sim4[0][0])
print("Similarity (Painting3 vs Mask1):", sim5[0][0])
print("Similarity (Painting3 vs Mask2):", sim6[0][0])

Similarity (Painting1 vs Mask1): 0.3472746
Similarity (Painting1 vs Mask2): 0.3270562
Similarity (Painting2 vs Mask1): 0.43513623
Similarity (Painting2 vs Mask2): 0.43276688
Similarity (Painting3 vs Mask1): 0.5038273
Similarity (Painting3 vs Mask2): 0.538597
